In [ ]:
from djitellopy import Tello
import cv2
import time
import os
from datetime import datetime

# Garbage categories
CATEGORIES = ['plastic', 'paper', 'metal', 'glass', 'organic', 'cardboard']

# Create dataset folders for each category
os.makedirs("dataset/images", exist_ok=True)
os.makedirs("dataset/videos", exist_ok=True)

for category in CATEGORIES:
    os.makedirs(f"dataset/images/{category}", exist_ok=True)

# Current category selection
current_category = CATEGORIES[0]  # Start with 'plastic'
category_index = 0

tello = Tello()
tello.connect()
print("Battery:", tello.get_battery())
tello.streamon()
time.sleep(3)

# Use OpenCV VideoCapture directly
cap = cv2.VideoCapture('udp://0.0.0.0:11111', cv2.CAP_FFMPEG)

if not cap.isOpened():
    print("Error: Could not open video stream")
    tello.end()
    exit()

# Video recording variables
recording = False
video_writer = None
frame_width = 960
frame_height = 720

# Photo counter for each category
photo_counters = {cat: 0 for cat in CATEGORIES}

print("\n" + "="*60)
print("TELLO GARBAGE DATASET COLLECTION")
print("="*60)
print("\nControls:")
print("'s' - Save photo to current category")
print("'n' - Next category")
print("'p' - Previous category")
print("'1-6' - Direct category selection")
print("'r' - Start/Stop recording")
print("'q' - Quit")
print("\nCategories:")
for i, cat in enumerate(CATEGORIES, 1):
    print(f"  {i}. {cat}")
print("="*60)

while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        continue
    
    # Resize frame for consistent size
    frame = cv2.resize(frame, (frame_width, frame_height))
    
    # Display current category (large box at top)
    cv2.rectangle(frame, (10, 10), (600, 100), (0, 0, 0), -1)
    cv2.rectangle(frame, (10, 10), (600, 100), (0, 255, 0), 3)
    cv2.putText(frame, f"Current Category: {current_category.upper()}", 
                (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
    cv2.putText(frame, f"Photos in {current_category}: {photo_counters[current_category]}", 
                (20, 85), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    
    # Show recording indicator
    if recording:
        cv2.circle(frame, (30, 120), 10, (0, 0, 255), -1)
        cv2.putText(frame, "REC", (50, 130), cv2.FONT_HERSHEY_SIMPLEX, 
                    0.7, (0, 0, 255), 2)
    
    # Display all category counts (sidebar)
    y_offset = 120
    cv2.rectangle(frame, (frame_width - 250, 10), (frame_width - 10, 250), (0, 0, 0), -1)
    cv2.putText(frame, "Dataset Summary:", (frame_width - 240, 35), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    for i, cat in enumerate(CATEGORIES):
        color = (0, 255, 0) if cat == current_category else (150, 150, 150)
        text = f"{i+1}. {cat}: {photo_counters[cat]}"
        cv2.putText(frame, text, (frame_width - 235, y_offset), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        y_offset += 35
    
    # Display battery
    cv2.putText(frame, f"Battery: {tello.get_battery()}%", 
                (10, frame_height - 10), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    
    # Display controls reminder
    cv2.putText(frame, "Press 'n' for next category | 's' to save", 
                (frame_width - 500, frame_height - 10), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    cv2.imshow("Tello Dataset Collector", frame)
    
    # Write frame if recording
    if recording and video_writer is not None:
        video_writer.write(frame)
    
    key = cv2.waitKey(1) & 0xFF
    
    # Save photo to current category (press 's')
    if key == ord('s'):
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        photo_counters[current_category] += 1
        filename = f"dataset/images/{current_category}/{current_category}_{photo_counters[current_category]:04d}_{timestamp}.jpg"
        cv2.imwrite(filename, frame)
        print(f"Photo saved: {filename}")
        print(f"Total {current_category} images: {photo_counters[current_category]}")
    
    # Next category (press 'n')
    elif key == ord('n'):
        category_index = (category_index + 1) % len(CATEGORIES)
        current_category = CATEGORIES[category_index]
        print(f"\nSwitched to category: {current_category}")
    
    # Previous category (press 'p')
    elif key == ord('p'):
        category_index = (category_index - 1) % len(CATEGORIES)
        current_category = CATEGORIES[category_index]
        print(f"\nSwitched to category: {current_category}")
    
    # Direct category selection (press '1'-'6')
    elif key in [ord('1'), ord('2'), ord('3'), ord('4'), ord('5'), ord('6')]:
        idx = int(chr(key)) - 1
        if idx < len(CATEGORIES):
            category_index = idx
            current_category = CATEGORIES[category_index]
            print(f"\nSwitched to category: {current_category}")
    
    # Start/Stop recording (press 'r')
    elif key == ord('r'):
        if not recording:
            # Start recording
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"dataset/videos/video_{timestamp}.avi"
            fourcc = cv2.VideoWriter_fourcc(*'XVID')
            video_writer = cv2.VideoWriter(filename, fourcc, 30.0, 
                                          (frame_width, frame_height))
            recording = True
            print(f"Recording started: {filename}")
        else:
            # Stop recording
            recording = False
            if video_writer is not None:
                video_writer.release()
                video_writer = None
            print("Recording stopped")
    
    # Quit (press 'q')
    elif key == ord('q'):
        break

# Cleanup
if recording and video_writer is not None:
    video_writer.release()

cap.release()
cv2.destroyAllWindows()
tello.streamoff()
tello.end()

# Print final summary
print("\n" + "="*60)
print("SESSION SUMMARY")
print("="*60)
print("\nDataset collected:")
total_images = 0
for cat in CATEGORIES:
    count = photo_counters[cat]
    total_images += count
    print(f"  {cat}: {count} images")
print(f"\nTotal images collected: {total_images}")
print("\nFiles saved in 'dataset/images/' organized by category")
print("="*60)